# AI4I 2020 Predictive Maintenance – Classification Project
This notebook solves the predictive maintenance classification problem using multiple ML techniques.

Methods included:
- Data preprocessing
- Feature engineering
- Handling imbalanced dataset (SMOTE)
- Feature scaling
- Feature selection
- Dimensionality reduction (PCA)
- Multiple ML models
- Hyperparameter tuning
- 5-fold cross validation
- Mean and standard deviation across 5 runs


In [9]:
!pip install xgboost

In [10]:
!pip install catboost

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

import xgboost as xgb
import catboost as cb


## Load Dataset

In [12]:
df = pd.read_csv('Final_Classification_Data.csv')
df.head()

,Unnamed: 0,Air_temperature_[K],Process_temperature_[K],Rotational_speed_[rpm],Torque_[Nm],Tool_wear_[min],Temp_Delta,Power_[W],Wear_per_Torque,Speed_Torque_Ratio,Type_H,Type_L,Type_M,Machine_failure
0,0,298.1,308.6,1551,42.8,0,10.5,6951.590560,0.000000,36.238309,0,0,1,0
1,1,298.2,308.7,1408,46.3,3,10.5,6826.722724,0.064795,30.410361,0,1,0,0
2,2,298.1,308.5,1498,49.4,5,10.4,7749.387543,0.101215,30.323881,0,1,0,0
3,3,298.2,308.6,1433,39.5,7,10.4,5927.504659,0.177215,36.278472,0,1,0,0
4,4,298.2,308.7,1408,40.0,9,10.5,5897.816608,0.225000,35.199991,0,1,0,0


## Separate Features and Target

In [13]:
X = df.drop('Machine_failure', axis=1)
y = df['Machine_failure']

## Train Validation Test Split (60-20-20)

In [14]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)

Train size: (6000, 13)
Validation size: (2000, 13)
Test size: (2000, 13)


## Define Preprocessing Components

In [15]:
scaler = StandardScaler()

selector = SelectKBest(score_func=f_classif, k=8)

pca = PCA(n_components=6)

smote = SMOTE(random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Define Models

In [19]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM': SVC(),
    'RandomForest': RandomForestClassifier(),
    'ExtraTrees': ExtraTreesClassifier(),
    'MLP': MLPClassifier(max_iter=2000),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss'),
    'CatBoost': cb.CatBoostClassifier(verbose=0)
}

## Run Each Model 5 Times

In [20]:
results = {}

for name, model in models.items():
    acc_scores = []

    for i in range(5):
        pipeline = Pipeline([
            ('scaler', scaler),
            ('smote', smote),
            ('selector', selector),
            ('pca', pca),
            ('model', model)
        ])

        pipeline.fit(X_train, y_train)

        preds = pipeline.predict(X_test)

        acc = accuracy_score(y_test, preds)

        acc_scores.append(acc)

    results[name] = {
        'mean_accuracy': np.mean(acc_scores),
        'std_accuracy': np.std(acc_scores)
    }

## Display Results

In [21]:
results_df = pd.DataFrame(results).T
results_df

,mean_accuracy,std_accuracy
LogisticRegression,0.7960,0.000000e+00
SVM,0.9220,0.000000e+00
RandomForest,0.9570,1.673320e-03
ExtraTrees,0.9609,5.830952e-04
MLP,0.9472,3.749667e-03
XGBoost,0.9505,1.110223e-16
CatBoost,0.9520,0.000000e+00
